# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, inspecting, and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is defined by a Croissant schema, accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets and their `@id`s, as well as fields (columns) within them, referencing all by their `@id` as per the FAIR² Croissant schema.

**Note:** All Croissant entities are referenced by their unique `@id`. This ensures reproducible and unambiguous selection of fields or record sets.

In [ ]:
# List all record sets and their columns by @id
print("Record sets:")
record_sets = list(dataset.list_record_sets())  # Returns a list of dicts with @id and name
for rs in record_sets:
    print(f"- Record set name: {rs['name']} | @id: {rs['@id']}")
    columns = dataset.list_fields(record_set=rs['@id'])  # List fields for each record set
    for col in columns:
        print(f"    - Field: {col['name']} | @id: {col['@id']}")

# For demonstration, preview a few records from the first record set
if record_sets:
    first_rs_id = record_sets[0]['@id']
    print(f"\nPreviewing first 2 records from record set (@id): {first_rs_id}")
    for i, record in enumerate(dataset.records(record_set=first_rs_id)):
        print(record)
        if i >= 1:
            break

## 3. Data Extraction

Load data from all record sets into DataFrames for analysis. We use the record set and field `@id`s referenced above.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns and few rows from the first record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Record set columns (@id): {list(dataframes[main_rs_id].columns)}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering, normalizing, and grouping for a numeric field. All field access is by `@id`.

In [ ]:
# For EDA, select (by @id) a numeric field and grouping field from the main record set (first in list)
main_df = dataframes[main_rs_id]

# Inspect available fields (columns) by @id and infer suitable ones for numeric and grouping
print("Columns available in main record set:")
for i, col in enumerate(main_df.columns):
    print(f"  {i}. {col}")

# Example (replace with actual @ids as present in the dataset):
# Let's define possible field ids. Example names (to be replaced accordingly):
# numeric_field_id = '@id_of_numeric_field' e.g. 'schema:age'
# group_field_id = '@id_of_grouping_category' e.g. 'schema:sex'

# Try to guess common IDs/names for numeric and group field

# Attempt to find a numeric column (e.g., age, diagnosis interval, etc.)
import numpy as np
numeric_field_id = None
group_field_id = None

# Select first float/int column as numeric; and a likely category one as group
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break

# Select a different column with object type as grouping (if available)
for col in main_df.columns:
    if col != numeric_field_id and main_df[col].dtype == 'object':
        group_field_id = col
        break

print(f"\nSelected numeric field for EDA: {numeric_field_id}")
print(f"Selected group field for EDA: {group_field_id}\n")

# Proceed only if numeric field is available
if numeric_field_id is not None:
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().any() else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}):")
    display(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )

    print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If group field available, group and aggregate
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field detected for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to the grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

if numeric_field_id is not None and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field is available, show its distribution
    if group_field_id is not None and group_field_id in main_df.columns:
        plt.figure(figsize=(9, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, you explored the FAIR² dataset defined by its Croissant schema and referenced each entity by its stable `@id`.

- We loaded dataset metadata and inspected available record sets and fields (all referenced by `@id`).
- We extracted tabular records for key record sets and performed simple filtering and grouping using a representative numeric and categorical field.
- Visualizations provided quick insights into value distributions and group effects.

You can adapt this notebook to perform further analyses by selecting or filtering on different record set `@id`s and fields annotated in the Croissant metadata.
